# GigBid evaluations on a free Kaggle GPU

Runs `make eval` with a **local open model** (Ollama, `gpt-oss:20b`), so there is no API key, no rate limit and no cost.

Before you run it, in the right-hand panel set:
- **Accelerator:** GPU T4 x2
- **Internet:** On

The repo must be pushed to GitHub first, because this notebook clones it.

Cells 3 to 5 set up and test the model first. If the model cannot be downloaded or cannot finish one job, the notebook
stops there, within about ten minutes, instead of failing after the long steps.

In [ ]:
# 1. Get the code
REPO = "https://github.com/aimennnnn/gigbid.git"
!rm -rf /kaggle/working/gigbid
!git clone --depth 1 $REPO /kaggle/working/gigbid
%cd /kaggle/working/gigbid
!git log --oneline -1

In [ ]:
# 2. Install the project (uv reads pyproject.toml and uv.lock, the same versions as on the laptop)
!pip install -q uv
!uv sync --frozen

In [ ]:
# 3. Install Ollama, start it in the background and download the model (about 14 GB)
import os
import subprocess
import time
!apt-get install -y -qq pciutils lshw zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
# a bigger context window than Ollama's default, so long prompts (job + evidence + schema) are not cut off
server = subprocess.Popen(["ollama", "serve"], env=os.environ | {"OLLAMA_CONTEXT_LENGTH": "16384"},
                          stdout=open("/kaggle/working/ollama.log", "w"), stderr=subprocess.STDOUT)
for attempt in range(1, 11):          # the model registry sometimes answers "503": wait and try again
    time.sleep(15)
    pull = subprocess.run(["ollama", "pull", "gpt-oss:20b"], capture_output=True, text=True)
    if pull.returncode == 0:
        break
    print(f"download attempt {attempt} failed: {pull.stderr.strip()[-200:]}")
models = subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout
print(models)
assert "gpt-oss:20b" in models, "the model was not downloaded, so the LLM steps cannot run"

In [ ]:
# 4. Point GigBid at the local model. These names are read by agent/llm.py (see agent/config.yaml, provider "ollama").
os.environ["OLLAMA_URL"] = "http://localhost:11434/v1"
os.environ["GIGBID_PROVIDER"] = "ollama"

# the hardware goes into the results folder, because the report must state it for local models
!mkdir -p evals/results
!nvidia-smi --query-gpu=name,memory.total --format=csv | tee evals/results/kaggle_hardware.txt
!nproc | xargs echo "CPU cores:" | tee -a evals/results/kaggle_hardware.txt

In [ ]:
# 5. Quick check: one real job through the bid graph. It must end without a model error, or the notebook stops here.
check = subprocess.run(["uv", "run", "python", "-m", "agent.demo"], capture_output=True, text=True)
print(check.stdout[-4000:])
print(check.stderr[-2000:])
assert "status=" in check.stdout and "ERRORS" not in check.stdout, "the local model did not get one job through the graph"

In [ ]:
# 6. Retrieval: every RAG variant, the ablation, and the sweep of the V8 threshold. No LLM; about an hour on Kaggle's CPU.
!uv run python -m evals.runners.run_all --only rag
!uv run python -m evals.runners.rag_eval --sweep

In [ ]:
# 7. Decisions and proposals with GigSim replies, then the results table. This is the long step.
#    --decisions and --jobs are jobs per persona. If the session stops half way, run the next cell instead.
!uv run python -m evals.runners.run_all --only agent --decisions 20 --jobs 10

In [ ]:
# 7b. ONLY if step 7 stopped half way: carries on the proposal run without redoing finished jobs.
# !uv run python -m evals.runners.run_all --only agent --resume

In [ ]:
# 8. Make the two sheets that people fill in: 30 proposals for you to score (judge calibration)
#    and the blind A/B sheet for classmates.
!uv run python -m evals.runners.judge_calibration --sheet
!uv run python -m evals.runners.judge_calibration --pairs

In [ ]:
# 9. Pack everything for download. The zip appears under Output in the right-hand panel.
!cat evals/results/results_table.md
!zip -q -r /kaggle/working/gigbid_results.zip evals/results evals/datasets/judge_labels.csv evals/datasets/pairwise_panel.csv README.md
print("Download /kaggle/working/gigbid_results.zip and unzip it over your gigbid folder.")

## After the run (on the laptop)

1. Unzip `gigbid_results.zip` over the project folder. It brings `evals/results/`, the two sheets and the README with the table.
2. Open `evals/datasets/judge_labels.csv`, score each proposal 1 to 5 in `human_score` using the rubric in `agent/prompts/judge.md`.
3. `uv run python -m evals.runners.judge_calibration` writes Cohen's kappa to `evals/results/judge_calibration.json` (no model needed).
4. Commit `evals/results/`, `evals/datasets/judge_labels.csv` and `README.md`.